## 1. Bronze Subject Schema

**Question**

What is the structure and data type definition of the Bronze EDC subject dataset?

**Purpose**

Understand the raw Bronze subject structure before designing Silver normalization, type handling, data-quality rules, and CDC processing.

In [0]:
%sql
DESCRIBE TABLE clinical_trial_intelligence.bronze.edc_subjects;

### Result

The Bronze EDC subject table contains subject identifiers, study and site identifiers, demographic attributes, clinical attributes, subject lifecycle dates, status information, and source lineage metadata.

Key clinical date and numeric fields are already represented using appropriate Bronze data types, including `DATE` and `INT`.

### Design conclusion

The Bronze schema provides the required attributes for Silver subject processing. The Silver pipeline will normalize business identifiers and categorical values, standardize reference-driven attributes, apply clinical data-quality rules, and preserve source lineage for auditability.

## 2. Sample Bronze Subject Records

**Question**

What does an actual Bronze subject record look like, including clinical attributes and ingestion lineage metadata?

**Purpose**

Inspect representative source records to understand the values available for Silver transformation, data-quality validation, reference standardization, and CDC processing.

In [0]:
%sql

SELECT
    subject_id,
    study_id,
    site_id,
    screening_date,
    enrollment_date,
    subject_status,
    age,
    sex,
    baseline_condition_code,
    arm_code,
    randomization_date,
    informed_consent_date,
    discontinuation_date,
    discontinuation_reason,
    _source_file_name,
    _source_file_modification_ts,
    _ingestion_ts,
    _ingestion_date
FROM clinical_trial_intelligence.bronze.edc_subjects
LIMIT 20;

### Result

The Bronze sample contains subject identifiers, study/site assignments, lifecycle dates, demographic attributes, clinical attributes, and ingestion lineage metadata.

The sample also reveals source-data quality and standardization issues. Sex is represented using multiple raw formats such as `f`, `F`, `Female`, `FEMALE`, `m`, `M`, and `MALE`, with additional non-standard values also present. A missing `study_id` is visible, and an age value of `118` demonstrates the need for clinical range validation.

Lifecycle fields also vary according to subject status. For example, a `SCREEN_FAILED` subject may have a null `enrollment_date`.

### Design conclusion

Bronze records should not be promoted directly to Silver. The Silver subject pipeline must normalize categorical values, resolve reference-driven attributes, validate required identifiers and clinically reasonable values, and apply status-aware lifecycle rules.

Null values must be evaluated according to business context rather than treated universally as data-quality failures. Source lineage metadata should be retained for traceability and auditability.

## 3. Subject Source File Pattern

**Question**

Do the EDC subject files represent complete subject snapshots or incremental delta/change files?

**Purpose**

Compare record volumes across source files to determine the correct CDC ingestion strategy for the Silver subject pipeline.

In [0]:
%sql
SELECT
    _source_file_name,
    COUNT(*) AS row_count,
    COUNT(DISTINCT TRIM(subject_id)) AS distinct_subjects
FROM clinical_trial_intelligence.bronze.edc_subjects
GROUP BY _source_file_name
ORDER BY _source_file_name;

### Result

The source-file inventory shows a clear difference between the initial subject file and subsequent files.

`subjects_20260825.csv` contains 3,389 rows representing 3,359 distinct subjects, whereas each subsequent source file contains only 55 rows representing 55 distinct subjects.

The substantially smaller subsequent files indicate that they contain incremental subject changes rather than complete replacements of the subject population.

The initial file also contains more rows than distinct subject identifiers, indicating that duplicate or repeated subject records require separate investigation.

### Design conclusion

The EDC subject source must be treated as an incremental delta/change feed rather than a sequence of complete snapshots.

Therefore, the Silver subject pipeline should use `create_auto_cdc_flow()` rather than `create_auto_cdc_from_snapshot_flow()`.

Treating the later 55-row files as complete snapshots would incorrectly interpret thousands of subjects absent from those files as deletions.

Before finalizing the CDC key and sequence, repeated subject records in the source must be investigated.

## 4. Subjects Appearing Across Multiple Source Files

**Question**

Do individual subjects appear in more than one EDC source file?

**Purpose**

Determine whether the delta feed contains repeated subject identifiers across different source dates, indicating that subject attributes can change over time and therefore require CDC processing.

In [0]:
%sql

SELECT
    TRIM(subject_id) AS subject_id,
    COUNT(*) AS record_count,
    COUNT(DISTINCT _source_file_name) AS source_file_count,
    MIN(_source_file_name) AS first_source_file,
    MAX(_source_file_name) AS latest_source_file,
    COLLECT_SET(_source_file_name) AS source_files

FROM clinical_trial_intelligence.bronze.edc_subjects

WHERE subject_id IS NOT NULL
  AND TRIM(subject_id) <> ''

GROUP BY TRIM(subject_id)

HAVING COUNT(DISTINCT _source_file_name) > 1

ORDER BY
    source_file_count DESC,
    record_count DESC

LIMIT 20;

### Result

Multiple subjects appear across several dated EDC source files.

For example, subject `104-003-0067` appears in three source files spanning `subjects_20260825.csv` through `subjects_20260904.csv`. Similar patterns are observed for multiple other subject identifiers.

The repeated appearance of the same `subject_id` across different source files confirms that later files can contain subsequent records for subjects already present in the initial load.

### Design conclusion

`subject_id` represents the business entity that must be tracked across the incoming delta feed.

Because the same subject can reappear in later source files, the Silver pipeline requires CDC processing to order successive records for each subject.

The source-file chronology must therefore be evaluated next to identify a reliable `sequence_by` value for `create_auto_cdc_flow()`.

SCD Type 2 remains appropriate for preserving clinically meaningful subject changes, but the actual attribute changes will be verified separately rather than inferred only from repeated subject identifiers.

## 5. CDC Sequencing Investigation

**Question**

Which source attribute provides the correct logical ordering of subject change events?

**Purpose**

Compare ingestion timestamps, file modification timestamps, and the business date encoded in the source filename to determine the appropriate `sequence_by` column for Silver CDC processing.

In [0]:
%sql

SELECT
    TRIM(subject_id) AS subject_id,
    _source_file_name,

    TO_DATE(
        REGEXP_EXTRACT(
            _source_file_name,
            'subjects_(\\d{8})\\.csv',
            1
        ),
        'yyyyMMdd'
    ) AS source_snapshot_date,

    _source_file_modification_ts,
    _ingestion_ts,

    subject_status,
    enrollment_date,
    discontinuation_date,
    arm_code,
    age,
    sex

FROM clinical_trial_intelligence.bronze.edc_subjects

WHERE TRIM(subject_id) IN (
    '104-003-0067',
    '104-010-0065',
    '102-003-0078'
)

ORDER BY
    subject_id,
    source_snapshot_date;

### Result

`_ingestion_ts` is identical across the inspected subject records because the historical source files were processed during the same Bronze ingestion run. It therefore cannot establish the logical order of subject changes.

`_source_file_modification_ts` also does not reliably represent the clinical source chronology. For example, the modification timestamps of the files containing subject `102-003-0078` do not follow the chronological order encoded in the source filenames.

In contrast, the date encoded in `_source_file_name` provides a consistent logical progression of the source records.

The inspected records also demonstrate actual clinical and demographic changes across source files. Examples include changes from `ENROLLED` to `DISCONTINUED`, population of `discontinuation_date`, and changes in `age`.

### Design conclusion

The Silver subject CDC pipeline will derive `source_snapshot_date` from the `YYYYMMDD` component of `_source_file_name` and use this value as the logical CDC sequence.

Therefore:

- Business key: `subject_id`
- CDC sequence: `source_snapshot_date`
- CDC mechanism: `create_auto_cdc_flow()`
- History strategy: SCD Type 2

`_ingestion_ts` and `_source_file_modification_ts` will be retained as operational lineage metadata but will not control CDC sequencing.

## 6. Source Filename Date Parsing

**Question**

Can a valid logical source date be derived from every Bronze EDC subject filename?

**Purpose**

Verify that the `YYYYMMDD` component of `_source_file_name` can be parsed reliably before using the derived date as the `sequence_by` value in the Silver CDC flow.

In [0]:
%sql

SELECT
    COUNT(*) AS total_rows,

    COUNT_IF(
        TO_DATE(
            REGEXP_EXTRACT(
                _source_file_name,
                'subjects_(\\d{8})\\.csv',
                1
            ),
            'yyyyMMdd'
        ) IS NULL
    ) AS invalid_source_dates,

    COUNT(DISTINCT _source_file_name) AS source_file_count,

    MIN(
        TO_DATE(
            REGEXP_EXTRACT(
                _source_file_name,
                'subjects_(\\d{8})\\.csv',
                1
            ),
            'yyyyMMdd'
        )
    ) AS earliest_source_date,

    MAX(
        TO_DATE(
            REGEXP_EXTRACT(
                _source_file_name,
                'subjects_(\\d{8})\\.csv',
                1
            ),
            'yyyyMMdd'
        )
    ) AS latest_source_date

FROM clinical_trial_intelligence.bronze.edc_subjects;

### Result

The Bronze EDC subject dataset contains 3,829 records across 9 source files, covering source dates from 2026-08-25 through 2026-09-07.

All 3,829 records successfully produce a valid date from the `YYYYMMDD` component of `_source_file_name`.

The number of records with an invalid derived source date is 0.

### Design conclusion

The source filename provides a reliable source-date value for every Bronze subject record.

Therefore, `source_snapshot_date` can be safely derived from `_source_file_name` and used as the logical sequencing column for subject CDC processing.

This supports the Silver CDC configuration:

- Business key: `subject_id`
- Sequence column: `source_snapshot_date`
- CDC processing: `create_auto_cdc_flow()`
- History strategy: SCD Type 2

## 7. CDC Key and Sequence Uniqueness

**Question**

Does each `subject_id` have at most one record for each derived `source_snapshot_date`?

**Purpose**

Verify that the combination of `subject_id` and `source_snapshot_date` uniquely identifies a subject change event before using `source_snapshot_date` as the sequencing column in the Silver AUTO CDC flow.

In [0]:
%sql

WITH subject_versions AS (

    SELECT
        TRIM(subject_id) AS subject_id,

        TO_DATE(
            REGEXP_EXTRACT(
                _source_file_name,
                'subjects_(\\d{8})\\.csv',
                1
            ),
            'yyyyMMdd'
        ) AS source_snapshot_date

    FROM clinical_trial_intelligence.bronze.edc_subjects

    WHERE subject_id IS NOT NULL
      AND TRIM(subject_id) <> ''
)

SELECT
    subject_id,
    source_snapshot_date,
    COUNT(*) AS record_count

FROM subject_versions

GROUP BY
    subject_id,
    source_snapshot_date

HAVING COUNT(*) > 1

ORDER BY
    record_count DESC,
    subject_id;

### Result

No duplicate combinations of `subject_id` and `source_snapshot_date` were identified.

Each subject therefore has at most one change record for a given source date.

### Design conclusion

The combination of `subject_id` and `source_snapshot_date` uniquely identifies the ordering of subject change events in the available Bronze data.

Therefore, a multi-column tie-breaker is not required for the current dataset.

The Silver subject CDC design can use:

- Business key: `subject_id`
- Sequence column: `source_snapshot_date`
- CDC processing: `create_auto_cdc_flow()`
- History strategy: SCD Type 2

## 8. Initial Source File Subject-Key Investigation

**Question**

Why does `subjects_20260825.csv` contain 3,389 rows but only 3,359 distinct non-null subject identifiers?

**Purpose**

Determine whether the difference is caused by duplicate subject records, missing subject identifiers, blank identifiers, or another source-data condition before finalizing the Silver subject data-quality rules.

In [0]:
%sql

SELECT
    _source_file_name,

    COUNT(*) AS total_rows,

    COUNT_IF(subject_id IS NULL) AS null_subject_ids,

    COUNT_IF(
        subject_id IS NOT NULL
        AND TRIM(subject_id) = ''
    ) AS blank_subject_ids,

    COUNT_IF(
        subject_id IS NOT NULL
        AND TRIM(subject_id) <> ''
    ) AS populated_subject_ids,

    COUNT(
        DISTINCT CASE
            WHEN subject_id IS NOT NULL
             AND TRIM(subject_id) <> ''
            THEN TRIM(subject_id)
        END
    ) AS distinct_populated_subject_ids

FROM clinical_trial_intelligence.bronze.edc_subjects

GROUP BY _source_file_name

ORDER BY _source_file_name;

### Result

The difference between the 3,389 total records and 3,359 distinct subjects in `subjects_20260825.csv` is entirely explained by 30 records with a NULL `subject_id`.

There are:

- 3,389 total records
- 30 records with NULL `subject_id`
- 0 records with blank `subject_id`
- 3,359 records with populated `subject_id`
- 3,359 distinct populated subject identifiers

Therefore, no duplicate populated subject identifiers exist within the initial source file.

All subsequent delta files contain 55 populated and 55 distinct subject identifiers.

### Design conclusion

`subject_id` is suitable as the business key for valid subject records.

Records with a NULL or blank `subject_id` cannot participate safely in key-based CDC processing and must be rejected before the AUTO CDC flow.

The Silver subject pipeline will therefore:

1. Normalize blank identifiers to NULL.
2. Apply the `missing_subject_id` data-quality rule.
3. Route records with missing subject identifiers to the subject quarantine table.
4. Allow only valid records with a populated `subject_id` to enter the Silver AUTO CDC flow.

No additional subject-key deduplication is required based on the available Bronze data.

## 9. Sex Reference Mapping Coverage

**Question**

Can the distinct sex values present in the Bronze EDC subject data be consistently standardized using `ref_sex`?

**Purpose**

Compare normalized Bronze sex values against the Silver sex reference table to identify unmapped values and confirm that reference-driven standardization is appropriate for the subject pipeline.

In [0]:
%sql

WITH bronze_sex AS (

    SELECT
        UPPER(TRIM(sex)) AS normalized_sex,
        COUNT(*) AS bronze_record_count

    FROM clinical_trial_intelligence.bronze.edc_subjects

    WHERE sex IS NOT NULL
      AND TRIM(sex) <> ''

    GROUP BY
        UPPER(TRIM(sex))
),

sex_reference AS (

    SELECT
        UPPER(TRIM(raw_sex)) AS normalized_raw_sex,
        UPPER(TRIM(standard_sex)) AS standard_sex

    FROM clinical_trial_intelligence.silver.ref_sex

    WHERE raw_sex IS NOT NULL
      AND TRIM(raw_sex) <> ''
)

SELECT
    b.normalized_sex AS bronze_sex_value,
    b.bronze_record_count,
    COLLECT_SET(r.standard_sex) AS mapped_standard_values,
    COUNT(DISTINCT r.standard_sex) AS standard_value_count,

    CASE
        WHEN COUNT(r.normalized_raw_sex) = 0
            THEN 'UNMAPPED'
        WHEN COUNT(DISTINCT r.standard_sex) > 1
            THEN 'AMBIGUOUS'
        ELSE 'MAPPED'
    END AS mapping_status

FROM bronze_sex b

LEFT JOIN sex_reference r
    ON b.normalized_sex = r.normalized_raw_sex

GROUP BY
    b.normalized_sex,
    b.bronze_record_count

ORDER BY
    mapping_status DESC,
    b.normalized_sex;

### Result

The Bronze subject dataset contains six normalized representations of sex: `1`, `2`, `F`, `FEMALE`, `M`, and `MALE`.

All six values are successfully resolved through `ref_sex`.

Each normalized Bronze value maps to exactly one standardized value:

- `1` → `M`
- `M` → `M`
- `MALE` → `M`
- `2` → `F`
- `F` → `F`
- `FEMALE` → `F`

No Bronze sex values are unmapped, and no normalized Bronze value maps to multiple standardized values.

### Design conclusion

Reference-driven sex standardization is suitable for the Silver subject pipeline.

The pipeline will normalize the source sex value using `UPPER(TRIM(...))` and resolve it through `ref_sex`.

The standardized Silver domain is therefore:

- `M`
- `F`

The `UNMAPPED` handling and `unmappable_sex` data-quality rule will remain in the pipeline as a defensive control for future source values that are not represented in `ref_sex`.

## 10. Baseline Diagnosis Reference Coverage

**Question**

Do all populated `baseline_condition_code` values in the Bronze subject data resolve to a diagnosis in `ref_diagnosis`?

**Purpose**

Validate diagnosis reference coverage before deciding whether an unresolved baseline diagnosis should be treated as a Silver data-quality failure.

In [0]:
%sql

WITH bronze_diagnosis AS (

    SELECT
        UPPER(TRIM(baseline_condition_code))
            AS baseline_condition_code,

        COUNT(*) AS bronze_record_count

    FROM clinical_trial_intelligence.bronze.edc_subjects

    WHERE baseline_condition_code IS NOT NULL
      AND TRIM(baseline_condition_code) <> ''
      AND TRIM(baseline_condition_code) <> '-'

    GROUP BY
        UPPER(TRIM(baseline_condition_code))
),

diagnosis_reference AS (

    SELECT
        UPPER(TRIM(diagnosis_code))
            AS diagnosis_code,

        diagnosis_description

    FROM clinical_trial_intelligence.silver.ref_diagnosis

    WHERE diagnosis_code IS NOT NULL
      AND TRIM(diagnosis_code) <> ''
)

SELECT
    b.baseline_condition_code,
    b.bronze_record_count,

    COLLECT_SET(r.diagnosis_description)
        AS diagnosis_descriptions,

    COUNT(DISTINCT r.diagnosis_description)
        AS diagnosis_description_count,

    CASE
        WHEN COUNT(r.diagnosis_code) = 0
            THEN 'UNMAPPED'

        WHEN COUNT(DISTINCT r.diagnosis_description) > 1
            THEN 'AMBIGUOUS'

        ELSE 'MAPPED'
    END AS mapping_status

FROM bronze_diagnosis b

LEFT JOIN diagnosis_reference r
    ON b.baseline_condition_code = r.diagnosis_code

GROUP BY
    b.baseline_condition_code,
    b.bronze_record_count

ORDER BY
    mapping_status DESC,
    b.baseline_condition_code;

### Result

The Bronze subject dataset contains nine populated baseline diagnosis codes, from `D001` through `D009`.

All nine diagnosis codes successfully resolve through `ref_diagnosis`, and each code maps to exactly one diagnosis description.

No unmapped or ambiguous diagnosis mappings were identified.

### Design conclusion

Reference-driven diagnosis enrichment is appropriate for the Silver subject pipeline.

`baseline_condition_code` will remain the clinical source code, while `baseline_condition` will be derived from `ref_diagnosis`.

Although the current Bronze dataset has complete diagnosis-reference coverage, the Silver pipeline should retain defensive validation for future source records containing diagnosis codes that are not represented in the reference table.

## 11. Study Reference Integrity

**Question**

Do all populated `study_id` values in the Bronze subject data correspond to valid studies in `dim_study`?

**Purpose**

Validate subject-to-study referential integrity and determine whether missing or unknown study identifiers must be handled by the Silver subject data-quality rules.

In [0]:
%sql

WITH bronze_studies AS (

    SELECT
        TRIM(study_id) AS study_id,
        COUNT(*) AS bronze_record_count

    FROM clinical_trial_intelligence.bronze.edc_subjects

    WHERE study_id IS NOT NULL
      AND TRIM(study_id) <> ''

    GROUP BY
        TRIM(study_id)
),

study_reference AS (

    SELECT DISTINCT
        TRIM(study_id) AS study_id

    FROM clinical_trial_intelligence.silver.dim_study

    WHERE study_id IS NOT NULL
      AND TRIM(study_id) <> ''
)

SELECT
    b.study_id,
    b.bronze_record_count,

    CASE
        WHEN s.study_id IS NULL
            THEN 'UNKNOWN'
        ELSE 'VALID'
    END AS study_status

FROM bronze_studies b

LEFT JOIN study_reference s
    ON b.study_id = s.study_id

ORDER BY
    study_status DESC,
    b.study_id;

In [0]:
%sql

SELECT
    COUNT(*) AS total_rows,

    COUNT_IF(
        study_id IS NULL
        OR TRIM(study_id) = ''
        OR TRIM(study_id) = '-'
    ) AS missing_study_id_rows

FROM clinical_trial_intelligence.bronze.edc_subjects;

### Result

All populated `study_id` values in the Bronze subject dataset successfully resolve to `dim_study`.

The populated source data references six studies:

- `CT-101`
- `CT-102`
- `CT-103`
- `CT-104`
- `CT-105`
- `CT-106`

No unknown populated study identifiers were identified.

However, 17 of the 3,829 Bronze subject records have a missing or unusable `study_id`.

### Design conclusion

The populated subject-to-study relationships are referentially valid, but study identification is not complete across the entire Bronze dataset.

The Silver subject pipeline will therefore retain both study data-quality controls:

- `missing_study_id` for records without a usable study identifier.
- `unknown_study_id` as a defensive control for future identifiers that do not resolve to `dim_study`.

Records failing these rules should be retained in the subject quarantine dataset rather than entering the validated Silver subject history.

## 12. Site and Study Relationship Integrity

**Question**

Do subject site identifiers resolve to `dim_site`, and does each resolved site belong to the same study referenced by the subject?

**Purpose**

Validate both site existence and the subject-site-study relationship before enforcing these relationships in the Silver subject pipeline.

In [0]:
%sql

WITH bronze_subjects AS (

    SELECT
        TRIM(subject_id) AS subject_id,
        TRIM(study_id) AS study_id,
        TRIM(site_id) AS site_id

    FROM clinical_trial_intelligence.bronze.edc_subjects
),

sites AS (

    SELECT DISTINCT
        TRIM(site_id) AS site_id,
        TRIM(study_id) AS site_study_id

    FROM clinical_trial_intelligence.silver.dim_site
)

SELECT

    COUNT(*) AS total_rows,

    COUNT_IF(
        b.site_id IS NULL
        OR b.site_id = ''
        OR b.site_id = '-'
    ) AS missing_site_id_rows,

    COUNT_IF(
        b.site_id IS NOT NULL
        AND b.site_id <> ''
        AND b.site_id <> '-'
        AND s.site_id IS NULL
    ) AS unknown_site_id_rows,

    COUNT_IF(
        s.site_id IS NOT NULL
        AND b.study_id IS NOT NULL
        AND b.study_id <> ''
        AND b.study_id <> '-'
        AND s.site_study_id <> b.study_id
    ) AS site_not_in_study_rows

FROM bronze_subjects b

LEFT JOIN sites s
    ON b.site_id = s.site_id;

### Result

The Bronze subject dataset contains 3,829 records.

Site relationship validation identified:

- 9 records with a missing or unusable `site_id`.
- 24 records with a populated `site_id` that does not resolve to `dim_site`.
- 0 records where a resolved site belongs to a different study than the `study_id` recorded for the subject.

Therefore, the available source data contains both missing and unknown site identifiers, while all successfully resolved site-to-study relationships are internally consistent.

### Design conclusion

The Silver subject pipeline must enforce site-level referential integrity before records enter the validated subject history.

The following data-quality controls are required:

- `missing_site_id` — the subject does not contain a usable site identifier.
- `unknown_site_id` — the supplied site identifier does not exist in `dim_site`.
- `site_not_in_study` — retained as a defensive control to detect future cases where a valid site is associated with the wrong study.

Records failing these controls should be retained in the subject quarantine dataset rather than entering the validated Silver SCD Type 2 history.

## 13. Study Arm Relationship Integrity

**Question**

Are subject arm assignments valid for the study to which each subject belongs?

**Purpose**

Validate the `(study_id, arm_code)` relationship against `dim_study_arm` and quantify records that require an arm assignment based on subject status but do not contain one.

In [0]:
%sql

WITH bronze_subjects AS (

    SELECT
        TRIM(study_id) AS study_id,
        UPPER(TRIM(subject_status)) AS subject_status,

        CASE
            WHEN arm_code IS NULL
              OR TRIM(arm_code) = ''
              OR TRIM(arm_code) = '-'
            THEN NULL
            ELSE UPPER(TRIM(arm_code))
        END AS arm_code

    FROM clinical_trial_intelligence.bronze.edc_subjects
),

study_arms AS (

    SELECT DISTINCT
        TRIM(study_id) AS study_id,
        UPPER(TRIM(arm_code)) AS arm_code

    FROM clinical_trial_intelligence.silver.dim_study_arm
)

SELECT
    COUNT(*) AS total_rows,

    COUNT_IF(
        b.subject_status IN (
            'ENROLLED',
            'DISCONTINUED',
            'COMPLETED'
        )
        AND b.arm_code IS NULL
    ) AS required_arm_missing_rows,

    COUNT_IF(
        b.arm_code IS NOT NULL
        AND a.arm_code IS NULL
    ) AS unknown_arm_rows

FROM bronze_subjects b

LEFT JOIN study_arms a
    ON b.study_id = a.study_id
   AND b.arm_code = a.arm_code;

### Result

The Bronze subject dataset contains 3,829 records.

- 40 records have a missing arm assignment even though the subject status indicates that an arm assignment is expected.
- 24 records contain an arm assignment that does not match a valid `(study_id, arm_code)` combination in `dim_study_arm`.

These results confirm that study-arm integrity issues are present in the Bronze subject data.

### Design conclusion

The Silver subject pipeline must validate study-arm assignments against `dim_study_arm` using the normalized `(study_id, arm_code)` combination.

Missing required arm assignments and unknown study-arm combinations should be treated as data-quality failures rather than silently accepted or removed. These conditions should therefore contribute to the subject-level DQ status and failure-reason metadata while preserving the source records for auditability.

## 14. Clinical Date Consistency

### Question

Are the clinical lifecycle dates in the Bronze subject records logically consistent?

### Purpose

Validate the chronological relationships between screening, enrollment, randomization, informed consent, and discontinuation dates before defining the Silver subject data-quality rules.

The objective is to identify records containing impossible or inconsistent clinical date sequences that should be flagged during Silver processing.

In [0]:
%sql

SELECT
    COUNT(*) AS total_rows,

    COUNT_IF(
        informed_consent_date IS NOT NULL
        AND screening_date IS NOT NULL
        AND informed_consent_date > screening_date
    ) AS consent_after_screening_rows,

    COUNT_IF(
        screening_date IS NOT NULL
        AND enrollment_date IS NOT NULL
        AND screening_date > enrollment_date
    ) AS screening_after_enrollment_rows,

    COUNT_IF(
        enrollment_date IS NOT NULL
        AND randomization_date IS NOT NULL
        AND enrollment_date > randomization_date
    ) AS enrollment_after_randomization_rows,

    COUNT_IF(
        enrollment_date IS NOT NULL
        AND discontinuation_date IS NOT NULL
        AND enrollment_date > discontinuation_date
    ) AS enrollment_after_discontinuation_rows,

    COUNT_IF(
        randomization_date IS NOT NULL
        AND discontinuation_date IS NOT NULL
        AND randomization_date > discontinuation_date
    ) AS randomization_after_discontinuation_rows

FROM clinical_trial_intelligence.bronze.edc_subjects;

### Result

Clinical date-sequence validation was performed across all 3,829 Bronze subject records.

No records were identified with:

- screening occurring after enrollment,
- enrollment occurring after randomization,
- enrollment occurring after discontinuation, or
- randomization occurring after discontinuation.

However, 27 records have an `informed_consent_date` later than the corresponding `screening_date`.

### Design conclusion

The Bronze subject data is largely consistent with the expected clinical lifecycle chronology, but a specific informed-consent timing issue is present.

The Silver subject pipeline should therefore retain temporal consistency checks and explicitly validate the relationship between informed consent and screening.

Records violating required clinical chronology should be captured as data-quality failures and routed to quarantine rather than entering the validated Silver subject history.

## 15. Final Subject Data-Quality Profile

### Question

What are the remaining subject-level data-quality conditions that must be handled by the Silver subject pipeline?

### Purpose

Quantify the core subject-level quality issues related to age, subject status, required screening dates, and informed-consent chronology before finalizing the Silver transformation and quarantine rules.

In [0]:
%sql

SELECT
    COUNT(*) AS total_rows,

    COUNT_IF(
        age IS NULL
        OR age < 18
        OR age > 100
    ) AS age_out_of_range_rows,

    COUNT_IF(
        subject_status IS NULL
        OR TRIM(subject_status) = ''
        OR UPPER(TRIM(subject_status)) NOT IN (
            'SCREENING',
            'ENROLLED',
            'SCREEN_FAILED',
            'DISCONTINUED',
            'COMPLETED'
        )
    ) AS invalid_subject_status_rows,

    COUNT_IF(
        screening_date IS NULL
    ) AS missing_screening_date_rows,

    COUNT_IF(
        informed_consent_date IS NOT NULL
        AND enrollment_date IS NOT NULL
        AND informed_consent_date > enrollment_date
    ) AS consent_after_enrollment_rows

FROM clinical_trial_intelligence.bronze.edc_subjects;

### Result

The final subject-level data-quality profile was evaluated across all 3,829 Bronze subject records.

- 24 records contain an age outside the accepted range of 18–100 years or a missing age.
- No records contain an invalid or missing subject status.
- No records are missing a screening date.
- 27 records have an informed-consent date later than the enrollment date.

The results confirm that age validity and informed-consent chronology represent active data-quality issues in the Bronze subject data, while subject status and screening-date completeness are currently clean.

### Design conclusion

The Silver subject pipeline should enforce explicit data-quality rules for age validity, subject-status validity, required screening dates, and clinical date chronology.

Records with invalid age values or informed-consent dates occurring after enrollment should be classified as data-quality failures. Although no current records violate the subject-status or screening-date rules, these checks should remain in the pipeline to protect against future source-data regressions.

Invalid records should be preserved with their failure reasons in the subject quarantine path rather than silently discarded, while valid records should proceed to the Silver SCD Type 2 subject history.

## 16. Consolidated Subject DQ Impact

### Question

How many Bronze subject records pass all identified subject-level data-quality rules, how many records violate at least one rule, and how much overlap exists between individual DQ findings?

### Purpose

Quantify the overall impact of the identified data-quality issues without double-counting records that violate multiple rules, and establish the expected valid and quarantined populations for Silver implementation.

In [0]:
%sql

WITH bronze_subjects AS (

    SELECT
        *,

        -- ----------------------------------------------------
        -- Normalized business identifiers
        -- ----------------------------------------------------

        CASE
            WHEN subject_id IS NULL
              OR TRIM(subject_id) IN ('', '-')
            THEN NULL
            ELSE TRIM(subject_id)
        END AS n_subject_id,

        CASE
            WHEN study_id IS NULL
              OR TRIM(study_id) IN ('', '-')
            THEN NULL
            ELSE TRIM(study_id)
        END AS n_study_id,

        CASE
            WHEN site_id IS NULL
              OR TRIM(site_id) IN ('', '-')
            THEN NULL
            ELSE TRIM(site_id)
        END AS n_site_id,

        CASE
            WHEN arm_code IS NULL
              OR TRIM(arm_code) IN ('', '-')
            THEN NULL
            ELSE UPPER(TRIM(arm_code))
        END AS n_arm_code,

        UPPER(TRIM(subject_status)) AS n_subject_status,

        TRY_CAST(age AS INT) AS n_age,

        TRY_CAST(screening_date AS DATE)
            AS n_screening_date,

        TRY_CAST(enrollment_date AS DATE)
            AS n_enrollment_date,

        TRY_CAST(informed_consent_date AS DATE)
            AS n_informed_consent_date,

        TRY_CAST(discontinuation_date AS DATE)
            AS n_discontinuation_date,

        TO_DATE(
            REGEXP_EXTRACT(
                _source_file_name,
                'subjects_(\\d{8})\\.csv',
                1
            ),
            'yyyyMMdd'
        ) AS source_snapshot_date,

        UPPER(TRIM(sex)) AS n_raw_sex

    FROM clinical_trial_intelligence.bronze.edc_subjects
),


-- ============================================================
-- REFERENCE / DIMENSION KEYS
-- ============================================================

sex_ref AS (

    SELECT DISTINCT
        UPPER(TRIM(raw_sex)) AS raw_sex
    FROM clinical_trial_intelligence.silver.ref_sex
    WHERE raw_sex IS NOT NULL
      AND TRIM(raw_sex) <> ''
),

studies AS (

    SELECT DISTINCT
        TRIM(study_id) AS study_id
    FROM clinical_trial_intelligence.silver.dim_study
),

sites AS (

    SELECT DISTINCT
        TRIM(site_id) AS site_id,
        TRIM(study_id) AS study_id
    FROM clinical_trial_intelligence.silver.dim_site
),

study_arms AS (

    SELECT DISTINCT
        TRIM(study_id) AS study_id,
        UPPER(TRIM(arm_code)) AS arm_code
    FROM clinical_trial_intelligence.silver.dim_study_arm
),


-- ============================================================
-- RESOLVE REFERENCE / DIMENSION RELATIONSHIPS
-- ============================================================

resolved AS (

    SELECT
        b.*,

        sx.raw_sex AS matched_sex,

        st.study_id AS matched_study_id,

        si.site_id AS matched_site_id,
        si.study_id AS site_study_id,

        sa.arm_code AS matched_arm_code

    FROM bronze_subjects b

    LEFT JOIN sex_ref sx
        ON b.n_raw_sex = sx.raw_sex

    LEFT JOIN studies st
        ON b.n_study_id = st.study_id

    LEFT JOIN sites si
        ON b.n_site_id = si.site_id

    LEFT JOIN study_arms sa
        ON b.n_study_id = sa.study_id
       AND b.n_arm_code = sa.arm_code
),


-- ============================================================
-- APPLY ALL DQ RULES
-- ============================================================

dq_evaluation AS (

    SELECT
        *,

        ARRAY_COMPACT(
            ARRAY(

                -- Business key
                CASE
                    WHEN n_subject_id IS NULL
                    THEN 'missing_subject_id'
                END,

                -- Study
                CASE
                    WHEN n_study_id IS NULL
                    THEN 'missing_study_id'
                END,

                CASE
                    WHEN n_study_id IS NOT NULL
                     AND matched_study_id IS NULL
                    THEN 'unknown_study_id'
                END,

                -- Site
                CASE
                    WHEN n_site_id IS NULL
                    THEN 'missing_site_id'
                END,

                CASE
                    WHEN n_site_id IS NOT NULL
                     AND matched_site_id IS NULL
                    THEN 'unknown_site_id'
                END,

                CASE
                    WHEN matched_site_id IS NOT NULL
                     AND site_study_id <> n_study_id
                    THEN 'site_not_in_study'
                END,

                -- Age
                CASE
                    WHEN n_age IS NULL
                      OR n_age NOT BETWEEN 18 AND 100
                    THEN 'age_out_of_range'
                END,

                -- Sex
                CASE
                    WHEN matched_sex IS NULL
                    THEN 'unmappable_sex'
                END,

                -- CDC sequence
                CASE
                    WHEN source_snapshot_date IS NULL
                    THEN 'invalid_source_snapshot_date'
                END,

                -- Screening
                CASE
                    WHEN n_screening_date IS NULL
                    THEN 'missing_screening_date'
                END,

                -- Temporal consistency
                CASE
                    WHEN n_enrollment_date IS NOT NULL
                     AND n_screening_date IS NOT NULL
                     AND n_enrollment_date < n_screening_date
                    THEN 'enrollment_before_screening'
                END,

                CASE
                    WHEN n_informed_consent_date IS NOT NULL
                     AND n_enrollment_date IS NOT NULL
                     AND n_informed_consent_date >
                         n_enrollment_date
                    THEN 'consent_after_enrollment'
                END,

                CASE
                    WHEN n_discontinuation_date IS NOT NULL
                     AND n_enrollment_date IS NOT NULL
                     AND n_discontinuation_date <
                         n_enrollment_date
                    THEN 'discontinuation_before_enrollment'
                END,

                -- Subject status
                CASE
                    WHEN n_subject_status IS NULL
                      OR n_subject_status NOT IN (
                            'SCREENING',
                            'ENROLLED',
                            'SCREEN_FAILED',
                            'DISCONTINUED',
                            'COMPLETED'
                         )
                    THEN 'invalid_subject_status'
                END,

                -- Required arm
                CASE
                    WHEN n_subject_status IN (
                            'ENROLLED',
                            'DISCONTINUED',
                            'COMPLETED'
                         )
                     AND n_arm_code IS NULL
                    THEN 'enrolled_without_arm'
                END,

                -- Unknown arm
                CASE
                    WHEN n_arm_code IS NOT NULL
                     AND matched_arm_code IS NULL
                    THEN 'unknown_arm_code'
                END

            )
        ) AS dq_failures

    FROM resolved
)


-- ============================================================
-- FINAL CONSOLIDATED RESULT
-- ============================================================

SELECT

    COUNT(*) AS total_rows,

    COUNT_IF(
        SIZE(dq_failures) = 0
    ) AS valid_rows,

    COUNT_IF(
        SIZE(dq_failures) > 0
    ) AS invalid_rows,

    ROUND(
        100.0 *
        COUNT_IF(SIZE(dq_failures) = 0)
        / COUNT(*),
        2
    ) AS valid_pct,

    ROUND(
        100.0 *
        COUNT_IF(SIZE(dq_failures) > 0)
        / COUNT(*),
        2
    ) AS invalid_pct,

    COUNT_IF(
        SIZE(dq_failures) > 1
    ) AS multi_failure_rows,

    MAX(
        SIZE(dq_failures)
    ) AS max_failures_per_row

FROM dq_evaluation;

## 17. Subject DQ Failure Breakdown

### Question

Which data-quality rules contribute to the invalid Bronze subject population, and how many records fail each rule?

### Purpose

Quantify the contribution of each subject-level DQ rule to the quarantined population and provide an auditable baseline for validating the Silver subject pipeline.

In [0]:
%sql

WITH bronze_subjects AS (

    SELECT
        *,

        CASE
            WHEN subject_id IS NULL
              OR TRIM(subject_id) IN ('', '-')
            THEN NULL
            ELSE TRIM(subject_id)
        END AS clean_subject_id,

        CASE
            WHEN study_id IS NULL
              OR TRIM(study_id) IN ('', '-')
            THEN NULL
            ELSE TRIM(study_id)
        END AS clean_study_id,

        CASE
            WHEN site_id IS NULL
              OR TRIM(site_id) IN ('', '-')
            THEN NULL
            ELSE TRIM(site_id)
        END AS clean_site_id,

        CASE
            WHEN arm_code IS NULL
              OR TRIM(arm_code) IN ('', '-')
            THEN NULL
            ELSE UPPER(TRIM(arm_code))
        END AS clean_arm_code,

        UPPER(TRIM(subject_status)) AS clean_subject_status,

        TRY_CAST(age AS INT) AS clean_age

    FROM clinical_trial_intelligence.bronze.edc_subjects
),

studies AS (

    SELECT DISTINCT
        TRIM(study_id) AS study_id

    FROM clinical_trial_intelligence.silver.dim_study
),

sites AS (

    SELECT DISTINCT
        TRIM(site_id) AS site_id,
        TRIM(study_id) AS study_id

    FROM clinical_trial_intelligence.silver.dim_site
),

arms AS (

    SELECT DISTINCT
        TRIM(study_id) AS study_id,
        UPPER(TRIM(arm_code)) AS arm_code

    FROM clinical_trial_intelligence.silver.dim_study_arm
),

evaluated AS (

    SELECT

        b.*,

        s.study_id AS matched_study_id,

        si.site_id AS matched_site_id,
        si.study_id AS site_study_id,

        a.arm_code AS matched_arm_code

    FROM bronze_subjects b

    LEFT JOIN studies s
        ON b.clean_study_id = s.study_id

    LEFT JOIN sites si
        ON b.clean_site_id = si.site_id

    LEFT JOIN arms a
        ON b.clean_study_id = a.study_id
       AND b.clean_arm_code = a.arm_code
),

dq_evaluation AS (

    SELECT

        ARRAY_COMPACT(
            ARRAY(

                CASE
                    WHEN clean_subject_id IS NULL
                    THEN 'missing_subject_id'
                END,

                CASE
                    WHEN clean_study_id IS NULL
                    THEN 'missing_study_id'
                END,

                CASE
                    WHEN clean_study_id IS NOT NULL
                     AND matched_study_id IS NULL
                    THEN 'unknown_study_id'
                END,

                CASE
                    WHEN clean_site_id IS NULL
                    THEN 'missing_site_id'
                END,

                CASE
                    WHEN clean_site_id IS NOT NULL
                     AND matched_site_id IS NULL
                    THEN 'unknown_site_id'
                END,

                CASE
                    WHEN matched_site_id IS NOT NULL
                     AND site_study_id <> clean_study_id
                    THEN 'site_not_in_study'
                END,

                CASE
                    WHEN clean_age IS NULL
                      OR clean_age NOT BETWEEN 18 AND 100
                    THEN 'age_out_of_range'
                END,

                CASE
                    WHEN screening_date IS NULL
                    THEN 'missing_screening_date'
                END,

                CASE
                    WHEN informed_consent_date IS NOT NULL
                     AND enrollment_date IS NOT NULL
                     AND informed_consent_date > enrollment_date
                    THEN 'consent_after_enrollment'
                END,

                CASE
                    WHEN clean_subject_status IS NULL
                      OR clean_subject_status NOT IN (
                            'SCREENING',
                            'ENROLLED',
                            'SCREEN_FAILED',
                            'DISCONTINUED',
                            'COMPLETED'
                         )
                    THEN 'invalid_subject_status'
                END,

                CASE
                    WHEN clean_subject_status IN (
                            'ENROLLED',
                            'DISCONTINUED',
                            'COMPLETED'
                         )
                     AND clean_arm_code IS NULL
                    THEN 'enrolled_without_arm'
                END,

                CASE
                    WHEN clean_arm_code IS NOT NULL
                     AND matched_arm_code IS NULL
                    THEN 'unknown_arm_code'
                END

            )
        ) AS dq_failures

    FROM evaluated
)

SELECT
    failure_reason,
    COUNT(*) AS failed_record_count

FROM dq_evaluation

LATERAL VIEW EXPLODE(dq_failures) exploded
    AS failure_reason

GROUP BY failure_reason

ORDER BY
    failed_record_count DESC,
    failure_reason;

**END OF EXPLORATION**